# Project 2 — Personal Finance Dashboard
## Notebook 0: Synthetic Data Generation

**Goal:** Generate realistic 12-month personal expense dataset for SQL analysis and Tableau visualisation  
**Skills demonstrated:** faker, numpy, pandas, realistic data modeling

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import date, timedelta
import random
import os

fake = Faker()
np.random.seed(42)
random.seed(42)

In [ ]:
# Expense categories with realistic monthly budget targets and merchants
categories = {
    'Housing':       {'budget': 1800, 'merchants': ['Apt Rent', 'Home Depot', 'IKEA'], 'freq': 1},
    'Groceries':     {'budget': 500,  'merchants': ['Whole Foods', 'Trader Joe\'s', 'Stop & Shop', 'Costco'], 'freq': 8},
    'Dining Out':    {'budget': 300,  'merchants': ['Chipotle', 'Panera', 'Local Bistro', 'Starbucks', 'DoorDash'], 'freq': 12},
    'Transport':     {'budget': 250,  'merchants': ['Uber', 'Lyft', 'Gas Station', 'MBTA'], 'freq': 15},
    'Healthcare':    {'budget': 150,  'merchants': ['CVS Pharmacy', 'Blue Cross', 'Mass General'], 'freq': 2},
    'Entertainment': {'budget': 200,  'merchants': ['Netflix', 'Spotify', 'AMC Theatres', 'Steam'], 'freq': 5},
    'Shopping':      {'budget': 350,  'merchants': ['Amazon', 'Target', 'Macy\'s', 'Best Buy'], 'freq': 6},
    'Utilities':     {'budget': 180,  'merchants': ['Eversource', 'National Grid', 'Comcast', 'T-Mobile'], 'freq': 4},
}

MONTHLY_INCOME = 5500
START_DATE     = date(2024, 1, 1)
END_DATE       = date(2024, 12, 31)

rows = []
txn_id = 1

current = START_DATE
while current <= END_DATE:
    month_dates = [current + timedelta(days=d)
                   for d in range(0, 28) if (current + timedelta(days=d)).month == current.month]

    # Add income credit
    rows.append({
        'transaction_id': txn_id, 'date': current.replace(day=1).isoformat(),
        'category': 'Income', 'merchant': 'Employer Payroll',
        'amount': MONTHLY_INCOME, 'type': 'credit'
    })
    txn_id += 1

    for cat, cfg in categories.items():
        n_txns = cfg['freq'] + random.randint(-2, 3)
        n_txns = max(1, n_txns)
        budget_per_txn = cfg['budget'] / cfg['freq']

        for _ in range(n_txns):
            # Add seasonal variance (higher spending in Nov/Dec)
            seasonal = 1.2 if current.month in [11, 12] and cat in ['Shopping', 'Dining Out'] else 1.0
            amount = round(abs(np.random.normal(budget_per_txn * seasonal, budget_per_txn * 0.25)), 2)
            amount = max(1.0, amount)

            rows.append({
                'transaction_id': txn_id,
                'date': random.choice(month_dates).isoformat(),
                'category': cat,
                'merchant': random.choice(cfg['merchants']),
                'amount': amount,
                'type': 'debit'
            })
            txn_id += 1

    # Advance to next month
    if current.month == 12:
        current = current.replace(year=current.year + 1, month=1, day=1)
    else:
        current = current.replace(month=current.month + 1, day=1)

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M').astype(str)
df['year']  = df['date'].dt.year

os.makedirs('../data', exist_ok=True)
df.to_csv('../data/expenses_synthetic.csv', index=False)

print(f'Generated {len(df):,} transactions')
print(f'Date range: {df.date.min().date()} to {df.date.max().date()}')
print('\nCategory breakdown:')
print(df[df.type=='debit'].groupby('category')['amount'].agg(['sum','count']).round(2))

In [ ]:
# Also create monthly budget reference table
budget_rows = []
for month_num in range(1, 13):
    for cat, cfg in categories.items():
        budget_rows.append({'month': f'2024-{month_num:02d}', 'category': cat, 'budget_amount': cfg['budget']})

budget_df = pd.DataFrame(budget_rows)
budget_df.to_csv('../data/monthly_budgets.csv', index=False)
print(f'Budget reference: {len(budget_df)} rows')
budget_df.head(10)